In [7]:

transactions = [
    {"id": 101, "user": "Alice", "amount": 5000, "status": "SUCCESS"},
    {"id": 102, "user": "Bob", "amount": 1200, "status": "FAILED"},
    {"id": 103, "user": "Charlie", "amount": 7500, "status": "SUCCESS"},
    {"id": 104, "user": "David", "amount": -500, "status": "SUCCESS"},
    {"id": 105, "user": "Eva", "amount": 3000, "status": "SUCCESS"},
    {"id": 106, "user": "Frank", "amount": 900, "status": "PENDING"},
    {"id": 107, "user": "Grace", "amount": 12000, "status": "SUCCESS"},
    {"id": 108, "user": "Henry", "amount": 0, "status": "SUCCESS"},
    {"id": 109, "user": "Ivy", "amount": 4500, "status": "SUCCESS"},
    {"id": 110, "user": "Jack", "amount": 2000, "status": "FAILED"}
]


# COMPONENT A: Transaction Reader
def read_transactions(transactions):
    for transaction in transactions:
        print("Reading transaction...")
        yield transaction


# COMPONENT B: Transaction Validator
def validate_transactions(transactions):

    for t in transactions:

        # Ensure the transaction is a dictionary
        if not isinstance(t, dict):
            print("Rejected: Transaction must be a dictionary")
            continue

        errors = []

        # Validate transaction ID
        if t.get("id") is None:
            errors.append("Missing transaction ID")

        # Validate user
        if not t.get("user"):
            errors.append("Missing user")

        # Validate amount
        amount = t.get("amount")

        if (
            not isinstance(amount, (int, float))
            or isinstance(amount, bool)
            or amount <= 0
        ):
            errors.append("Invalid transaction amount")

        # Validate status
        if t.get("status") not in ["SUCCESS", "FAILED", "PENDING"]:
            errors.append("Invalid transaction status")

        # Reject invalid transactions
        if errors:
            print(f"Rejected transaction {t.get('id')}: {errors}")
            continue

        # Yield valid transaction
        yield t


# COMPONENT C: Successful Transaction Filter
def successful_transactions(transactions):

    for t in transactions:
        if t["status"] == "SUCCESS":
            yield t


# COMPONENT D: High-Value Transaction Filter
def high_value_transactions(transactions):

    for t in transactions:
        if t["amount"] >= 3000:
            yield t


# CONNECT ALL GENERATORS USING YIELD FROM
def transaction_pipeline(transactions):

    yield from high_value_transactions(
        successful_transactions(
            validate_transactions(
                read_transactions(transactions)
            )
        )
    )


# COMPONENT E: Transaction Report
def generate_report(transactions):

    iterator = iter(transactions)

    # Get the first transaction safely
    first = next(iterator, None)

    # Handle empty generator
    if first is None:
        return {
            "total_transactions": 0,
            "total_amount": 0,
            "highest_amount": 0,
            "lowest_amount": 0,
            "average_amount": 0
        }

    # Initialize statistics
    count = 1
    total = first["amount"]
    highest = first["amount"]
    lowest = first["amount"]

    # Process remaining transactions lazily
    for t in iterator:

        amount = t["amount"]

        count += 1
        total += amount

        if amount > highest:
            highest = amount

        if amount < lowest:
            lowest = amount

    average = total / count

    return {
        "total_transactions": count,
        "total_amount": total,
        "highest_amount": highest,
        "lowest_amount": lowest,
        "average_amount": average
    }


# MAIN PROGRAM
if __name__ == "__main__":

    print("\n========== FINAL TRANSACTIONS ==========\n")

    # Create and consume the pipeline
    for transaction in transaction_pipeline(transactions):
        print(transaction)

    print("\n========== TRANSACTION REPORT ==========\n")

    # Create a FRESH pipeline because generators are exhausted
    report = generate_report(transaction_pipeline(transactions))

    for key, value in report.items():
        print(f"{key}: {value}")



========== FINAL TRANSACTIONS ==========

Reading transaction...
{'id': 101, 'user': 'Alice', 'amount': 5000, 'status': 'SUCCESS'}
Reading transaction...
Reading transaction...
{'id': 103, 'user': 'Charlie', 'amount': 7500, 'status': 'SUCCESS'}
Reading transaction...
Rejected transaction 104: ['Invalid transaction amount']
Reading transaction...
{'id': 105, 'user': 'Eva', 'amount': 3000, 'status': 'SUCCESS'}
Reading transaction...
Reading transaction...
{'id': 107, 'user': 'Grace', 'amount': 12000, 'status': 'SUCCESS'}
Reading transaction...
Rejected transaction 108: ['Invalid transaction amount']
Reading transaction...
{'id': 109, 'user': 'Ivy', 'amount': 4500, 'status': 'SUCCESS'}
Reading transaction...

========== TRANSACTION REPORT ==========

Reading transaction...
Reading transaction...
Reading transaction...
Reading transaction...
Rejected transaction 104: ['Invalid transaction amount']
Reading transaction...
Reading transaction...
Reading transaction...
Reading transaction...
